# Notebook 01 — Tool Use & Function Calling

**Workshop:** Agentic AI for Scientists — Week 2
**Block:** 2 of 6 (25 min)
**Goal:** Convince yourself that "tool use" is an `if model_response.has_tool_call: run_tool(...)` line inside a `while` loop. No magic.

We build the same tiny agent **three** times, each less code than the last:

1. **Hand-rolled.** LLM call + a regex + a Python function. No SDK helpers — you see every moving part.
2. **Gemini native function-calling.** Same control flow, but the model emits typed function-call parts so we stop parsing text.
3. **LangChain tool-calling agent.** The whole loop collapses to `AgentExecutor.invoke(...)`. This is the **function-calling agent** from the lecture — tool selection is shifted to the model vendor.

Seeing all three back-to-back is the point: when a framework hides the loop, you'll know exactly what it hid.


## 0. Setup

In [ ]:
%pip install -q "google-genai>=1.0" \
    "langchain==0.3.7" "langchain-google-genai==2.0.11" python-dotenv


In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


In [ ]:
import re
from google import genai
from google.genai import types

client = genai.Client()        # reads GOOGLE_API_KEY from the environment
MODEL = "gemini-2.5-flash"


def show(turn: str, content: str) -> None:
    """Pretty-print one turn of the conversation."""
    print(f"\n--- {turn} ---")
    print(content)


def as_text(output) -> str:
    """An AgentExecutor result["output"] is a plain string for both ReAct and the
    Gemini tool-calling agent. Tiny normaliser, kept in case a model returns parts."""
    if isinstance(output, str):
        return output
    return "".join(getattr(b, "text", "") or "" for b in output)


print(f"Ready with model: {MODEL}")


---

## 1. Define a tool — plain Python, no schema

A "tool" is just a function the model can ask you to call. Nothing more.


In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the result as a string."""
    expression = expression.replace("^", "**")   # models often write ^ for "to the power of"
    if not re.fullmatch(r"[\d\s+\-*/().]+", expression):
        return "ERROR: only basic arithmetic allowed"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"ERROR: {exc}"


print(calculator("(17 + 25) * 3"))


---

## 2. Hand-built tool loop (no SDK helpers)

We tell the model: *"If you need math, output `CALL: calculator(<expression>)`. Otherwise answer directly."* Then we parse the response ourselves and loop.

This is the entire substance of tool use. Frameworks add type safety, JSON schemas, and retries — but the control flow is exactly this.


In [ ]:
SYSTEM_PROMPT_MANUAL = """You are a helpful research assistant.

You have access to ONE tool: a calculator.

To use the calculator, output EXACTLY this on its own line:
CALL: calculator(<expression>)

For example: CALL: calculator(17 * 25)

The user will run the calculation and reply with the result. Then continue your answer.
If you don't need the calculator, just answer directly."""


def run_manual_loop(user_question: str, max_turns: int = 5) -> str:
    contents = [types.Content(role="user", parts=[types.Part(text=user_question)])]
    show("USER", user_question)

    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT_MANUAL, max_output_tokens=1024)

    for turn in range(max_turns):
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=config,
        )
        assistant_text = response.text
        show(f"ASSISTANT (turn {turn + 1})", assistant_text)

        # Parse the text ourselves to find a tool call
        match = re.search(r"CALL:\s*calculator\((.+?)\)", assistant_text)
        if not match:
            return assistant_text  # no tool needed -> done

        expression = match.group(1).strip()
        result = calculator(expression)
        show("TOOL RESULT", f"{expression} = {result}")

        # Feed the tool result back and let the model continue.
        # Gemini's roles are "user" and "model" (not "assistant").
        contents.append(types.Content(role="model", parts=[types.Part(text=assistant_text)]))
        contents.append(types.Content(role="user",
            parts=[types.Part(text=f"Tool result: {result}\n\nContinue your answer.")]))

    return "Max turns reached without final answer."


final = run_manual_loop(
    "A bacterial colony doubles every 20 minutes. Starting with 500 cells, "
    "how many cells after 3 hours? Show the calculation."
)
print("\n=== FINAL ANSWER ===")
print(final)


**What just happened?**

1. We sent the question to the LLM.
2. The LLM responded with `CALL: calculator(500 * 2**9)` (or similar) — a *text* response we agreed to interpret.
3. We parsed that text with a regex, ran the Python function, captured the result.
4. We appended the result back as a `user` message and re-prompted.
5. The LLM read the result and produced a final answer.

That is the whole pattern: **the loop is a Python `for`, the tool is a function, the agent is the LLM-plus-loop together.** Everything after this is removing the fragile parts.

---

## 3. Same thing with Gemini's native function-calling API

The hand-rolled version works but is brittle (regex parsing, prompt instructions, injection risk). Gemini's typed function-call parts give the same control flow with parsing handled for us.


In [ ]:
# Tool schema as a Gemini FunctionDeclaration. The model sees this and emits
# typed function-call parts instead of free text we have to regex.
calculator_decl = types.FunctionDeclaration(
    name="calculator",
    description="Evaluate a basic arithmetic expression. Use whenever a calculation is needed.",
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "expression": types.Schema(
                type="STRING",
                description="A math expression using digits, +, -, *, /, parens, dots.",
            ),
        },
        required=["expression"],
    ),
)
TOOLS = [types.Tool(function_declarations=[calculator_decl])]

# Map name -> Python callable. This is your "tool registry".
TOOL_REGISTRY = {"calculator": calculator}


In [ ]:
def run_native_loop(user_question: str, max_turns: int = 5) -> str:
    contents = [types.Content(role="user", parts=[types.Part(text=user_question)])]
    show("USER", user_question)
    config = types.GenerateContentConfig(tools=TOOLS)

    for turn in range(max_turns):
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=config,
        )
        parts = response.candidates[0].content.parts
        text_parts = [p.text for p in parts if getattr(p, "text", None)]
        calls = [p.function_call for p in parts if getattr(p, "function_call", None)]

        if text_parts:
            show(f"ASSISTANT (turn {turn + 1})", "\n".join(text_parts))
        if not calls:
            return "\n".join(text_parts)

        # Append the model's full turn, then run each requested function and
        # send the results back as function_response parts.
        contents.append(types.Content(role="model", parts=parts))
        result_parts = []
        for call in calls:
            args = dict(call.args)
            result = TOOL_REGISTRY[call.name](**args)
            show(f"TOOL {call.name}({args})", str(result))
            result_parts.append(types.Part.from_function_response(
                name=call.name, response={"result": str(result)}))
        contents.append(types.Content(role="user", parts=result_parts))

    return "Max turns reached."


final = run_native_loop(
    "A bacterial colony doubles every 20 minutes. Starting with 500 cells, "
    "how many cells after 3 hours? Show the calculation."
)
print("\n=== FINAL ANSWER ===")
print(final)


**What changed?**

| | Hand-rolled | Native function-calling |
|---|---|---|
| Tool definition | Python function + prompt instructions | Python function + JSON schema |
| Parsing | Regex on response text | SDK gives you typed function-call parts |
| Loop | `for` loop | same `for` loop |
| Injection risk | Higher (model could fake a `CALL:` mid-text) | Lower (tool blocks are structurally separate) |
| Type safety | None | JSON Schema validates inputs |

**The control flow is identical.** Only the parsing got safer.

---

## 4. Same thing again, with LangChain — the *tool-calling agent*

Now we let LangChain own the loop. This is the **function-calling agent**: we hand the model a set of tools, and the *model vendor* decides which tool to call with which arguments. We supply the tools and the loop runs itself.


In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)


# @tool reads the type hints + docstring to build the schema automatically.
@tool
def calc(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '500 * 2 ** 9'."""
    return calculator(expression)


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Returns a short description."""
    # Mock implementation so the notebook runs offline.
    fake = {"singapore": "31C, humid, afternoon thunderstorms likely",
            "london": "12C, overcast, light rain"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")


lc_tools = [calc, get_weather]

# The tool-calling prompt MUST include an agent_scratchpad MessagesPlaceholder.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful research assistant. Use tools when they help."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),   # where the loop's tool calls/results accumulate
])

agent = create_tool_calling_agent(llm, lc_tools, prompt)
executor = AgentExecutor(agent=agent, tools=lc_tools, verbose=True, max_iterations=5)

result = executor.invoke({
    "input": "Should I bring an umbrella in Singapore today? Also, if a 5 mm umbrella "
             "doubled in size every hour, how big after 4 hours?",
})
print("\n=== FINAL ANSWER ===")
print(as_text(result["output"]))   # Gemini's tool-calling agent returns a plain string


Watch the `verbose=True` trace: the agent called `get_weather('Singapore')` **and** `calc('5 * 2 ** 4')`, then composed both results into one answer — and we never wrote a loop. That loop is the same `for` from sections 2 and 3, now living inside `AgentExecutor.invoke`.

**Where the responsibility went.** With the tool-calling agent, *which* tool to call and *what arguments* to pass is decided by the model (a capability the vendor fine-tuned and serves). Our job shrinks to two things: write tools with **unambiguous names and descriptions**, and register them. This is a shared-responsibility model — the vendor picks the tool, but only as well as your descriptions let it.

---

## 5. Structured output — function calling, aimed at the answer

The same function-calling machinery has a twin. Instead of pointing a schema at a *tool*, point it at the **answer**: define the exact shape you want back as a Pydantic model, and the model fills it in. You get a typed, validated Python object — not a string you have to parse. This is how you make an agent's output safe to feed into the next step of a pipeline.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class ClaimCheck(BaseModel):
    """A structured verdict about a factual claim."""
    verdict: Literal["true", "false", "uncertain"] = Field(description="Is the claim correct?")
    confidence: Literal["low", "medium", "high"]
    reason: str = Field(description="A short justification, <= 15 words.")


# .with_structured_output binds the schema to the model via function calling.
checker = llm.with_structured_output(ClaimCheck)

result = checker.invoke(
    "Claim: the Transformer paper 'Attention Is All You Need' was published in 2017."
)
print(type(result).__name__)            # ClaimCheck — a real Python object, not a string
print("verdict   :", result.verdict)
print("confidence:", result.confidence)
print("reason    :", result.reason)


No regex, no JSON parsing, no "please respond in JSON" prompt-begging. `result` is a validated `ClaimCheck` object — `result.verdict` is *guaranteed* to be one of the three literals or the call fails loudly. Under the hood it's the **same** function-calling mechanism as the tool-calling agent, which is why both live in this notebook: one fills a tool's arguments, the other fills your answer's fields.

---

## 6. Closing exercises (for after class)

1. **Add a third tool** — `convert_units(value: float, from_unit: str, to_unit: str)`. Notice that the `@tool` decorator turns the three typed arguments into a schema with no extra work. Ask a question that needs all three tools.
2. **Break it on purpose.** Give two tools nearly identical descriptions ("does math" / "computes numbers"). Does the agent still pick correctly? This is why description quality beats prompt length in agent-land.
3. **Typed extraction.** Make `ClaimCheck` hold a `list` of verdicts, then ask the model to pull every claim out of a paragraph and check each one. The result drops straight into Python with no parsing.

---

**Next:** [Notebook 02 — ReAct, Chain-of-Thought & Tree-of-Thoughts](02_react_agent.ipynb)
